This notebook uses `deployment_tricondnet.py` to train the final ensemble models on 100% of the data, using the optimal hyperparameters selected from the nested CV results.

In [ ]:
import pandas as pd
import sys 
from pathlib import Path
import pickle as pkl
import sys 
sys.path.insert(0, str(Path.cwd().parent))
from models import deployment_tricondnet

Generate the classifier ensemble using the selected hyperparameters.

In [ ]:
data_path = Path.cwd().parent / "data" 
cbfvs = ["magpie"] # You can add more CBFVs if you ran the nested CV for different CBFVs, results can be aggregated across CBFVs if you choose to!
Number_Of_Ensemble_Models = 20 # You can change this!
seeds = range(0, Number_Of_Ensemble_Models)
for cbfv in cbfvs:
    altered_data_path = data_path / f"{cbfv}_data.csv"
    gbm = deployment_tricondnet.GBM_ClassifierDeployment(data_path=altered_data_path, cbfv=cbfv, random_state=0, validation=False)
    gbm.load_hyperparameters(pickle_path_file=Path.cwd() / "saved_hyperparameters" / f"{cbfv}_hyperparameters" / "classifier_hp.pkl")
    gbm.feature_ranking(feature_rank=True, target_to_correlate="class")
    out_root = Path.cwd() / "models" / cbfv
    out_root.mkdir(parents=True, exist_ok=True)        # add this line
    gbm.ensemble_models(seeds=seeds, root_path=out_root)

Repeat for the regressor models!

Semiconductor regressor...

In [ ]:
for cbfv in cbfvs:
    altered_data_path = data_path / f"{cbfv}_data.csv"
    semi = deployment_tricondnet.Semiconductor_Deployment(data_path=altered_data_path, cbfv=cbfv, target_name="target", random_state=0, validation=False)
    semi.load_hp(hp_path=Path.cwd() / "saved_hyperparameters" / f"{cbfv}_hyperparameters" / "semi_hp.pkl")
    semi.feature_ranking(feature_rank=True)
    out_root = Path.cwd() / "models" / cbfv
    out_root.mkdir(parents=True, exist_ok=True)        # add this line
    semi.ensemble_models(seeds=seeds, model_name="SemiconductorEnsemble", root_path=out_root)

Metal regressor...

In [ ]:
for cbfv in cbfvs:
    altered_data_path = data_path / f"{cbfv}_data.csv"
    metal = deployment_tricondnet.Metal_Deployment(data_path=altered_data_path, cbfv=cbfv, target_name="target", random_state=0, validation=False)
    metal.load_hp(hp_path=Path.cwd() / "saved_hyperparameters" / f"{cbfv}_hyperparameters" / "metal_hp.pkl")
    metal.feature_ranking(feature_rank=True)
    out_root = Path.cwd() / "models" / cbfv
    out_root.mkdir(parents=True, exist_ok=True)        # add this line
    metal.ensemble_models(seeds=seeds, model_name="MetalEnsemble", root_path=out_root)